In [13]:
# ML Assignment - Data Preparation & Feature Engineering
# Dataset: Titanic

import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   

In [14]:
# PART 1 - Feature Engineering

# Task 1: New Features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100], labels=['Child','Teen','YoungAdult','Adult','Senior'])
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# Task 2: Text Features
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df['Name_Length'] = df['Name'].str.len()
df['HasCabin'] = df['Cabin'].notna().astype(int)

print(df[['FamilySize','IsAlone','AgeGroup','FarePerPerson','Title','HasCabin']].head(10))

   FamilySize  IsAlone    AgeGroup  FarePerPerson   Title  HasCabin
0           2        0  YoungAdult        3.62500      Mr         0
1           2        0       Adult       35.64165     Mrs         1
2           1        1  YoungAdult        7.92500    Miss         0
3           2        0  YoungAdult       26.55000     Mrs         1
4           1        1  YoungAdult        8.05000      Mr         0
5           1        1         NaN        8.45830      Mr         0
6           1        1       Adult       51.86250      Mr         1
7           5        0       Child        4.21500  Master         0
8           3        0  YoungAdult        3.71110     Mrs         0
9           2        0        Teen       15.03540     Mrs         0


In [15]:
# PART 2 - Feature Encoding

# Task 3: One-Hot Encoding with get_dummies
cat_cols = ['Sex', 'Embarked', 'Title', 'Pclass']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print("After get_dummies Shape:", df_encoded.shape)
print(df_encoded.head(3))

After get_dummies Shape: (891, 36)
   PassengerId  Survived                                               Name  \
0            1         0                            Braund, Mr. Owen Harris   
1            2         1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2            3         1                             Heikkinen, Miss. Laina   

    Age  SibSp  Parch            Ticket     Fare Cabin  FamilySize  ...  \
0  22.0      1      0         A/5 21171   7.2500   NaN           2  ...   
1  38.0      1      0          PC 17599  71.2833   C85           2  ...   
2  26.0      0      0  STON/O2. 3101282   7.9250   NaN           1  ...   

   Title_Miss Title_Mlle  Title_Mme  Title_Mr  Title_Mrs  Title_Ms  Title_Rev  \
0           0          0          0         1          0         0          0   
1           0          0          0         0          1         0          0   
2           1          0          0         0          0         0          0   

   Title_Sir  Pclass_2

In [16]:
# Task 4: ColumnTransformer (Recommended Way)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numerical_features = ['Age', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title']

X = df[numerical_features + categorical_features]
y = df['Survived']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features),
        ('num', 'passthrough', numerical_features)
    ])

X_transformed = preprocessor.fit_transform(X)
print("Original Shape:", X.shape)
print("Transformed Shape:", X_transformed.shape)

Original Shape: (891, 8)
Transformed Shape: (891, 29)


In [17]:
# PART 3 - Feature Scaling

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Task 5: StandardScaler
scaler = StandardScaler()
df_std = pd.DataFrame(scaler.fit_transform(df[['Age','Fare','FamilySize']]), columns=['Age','Fare','FamilySize'])

print("After StandardScaler:")
print("Mean:", df_std.mean().values) # ~0 aayega
print("Std:", df_std.std().values)   # ~1 aayega
print(df_std.head(3))

# Task 6: MinMaxScaler
minmax = MinMaxScaler()
df_mm = pd.DataFrame(minmax.fit_transform(df[['Age','Fare','FamilySize']]), columns=['Age','Fare','FamilySize'])

print("\nAfter MinMaxScaler:")
print("Min:", df_mm.min().values) # 0 aayega
print("Max:", df_mm.max().values) # 1 aayega
print(df_mm.head(3))

After StandardScaler:
Mean: [ 2.38837894e-16  3.98733297e-18 -2.39239978e-17]
Std: [1.00070102 1.00056164 1.00056164]
        Age      Fare  FamilySize
0 -0.530377 -0.502445    0.059160
1  0.571831  0.786845    0.059160
2 -0.254825 -0.488854   -0.560975

After MinMaxScaler:
Min: [0. 0. 0.]
Max: [1. 1. 1.]
        Age      Fare  FamilySize
0  0.271174  0.014151         0.1
1  0.472229  0.139136         0.1
2  0.321438  0.015469         0.0


In [18]:
# PART 4 + PART 5 - Full ML Pipeline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Numerical Pipeline -> Impute + Scaling
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline -> Impute + Encoding
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine
preprocessor_final = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Final Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_final),
    ('model', LogisticRegression(max_iter=1000))
])

# Fit + Predict
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8324022346368715
              precision    recall  f1-score   support

           0       0.85      0.88      0.87       110
           1       0.80      0.75      0.78        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

Confusion Matrix:
 [[97 13]
 [17 52]]


In [ ]:
# Graphs for Better Visualization

import matplotlib.pyplot as plt
import seaborn as sns

# Graph 1: Survival by Sex
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Sex', hue='Survived')
plt.title('Survival by Sex - Females survived more')
plt.show()

# Graph 2: Survival by Pclass
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Pclass', hue='Survived')
plt.title('Survival by Pclass - 1st Class survived more')
plt.show()

# Graph 3: Age Distribution
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='Age', hue='Survived', bins=20)
plt.title('Age Distribution by Survival')
plt.show()

# Results & Conclusion

## Task 7: Scaling Comparison
- **StandardScaler:** Data ka Mean ~0 aur Std ~1 kar deta hai. Ye Logistic Regression jaise models ke liye best hai kyunki wo distance aur gradient par kaam karte hai.
- **MinMaxScaler:** Data ko 0 se 1 ke beech me la deta hai. Ye tab best hai jab hume fixed range chahiye.

**Conclusion:** Is dataset ke liye StandardScaler better hai kyunki Fare me bahut bade outliers hai, MinMax unse affect ho jata hai.

## Task 8: When to use get_dummies vs OneHotEncoder?
- **get_dummies:** Sirf quick EDA / small analysis ke liye. Test data me naya category aaya to fail ho jayega.
- **OneHotEncoder + ColumnTransformer:** Real Project / Pipeline ke liye. Ye Train aur Test par same transformation apply karta hai, isliye Production me yehi use hota hai.

## Task 9: Feature Importance
Top Features jo Survival ko sabse jyada affect karti hai:
1. **Sex (Female):** Sabse important feature - Females ke bachne ke chances sabse jyada the.
2. **Pclass:** 1st Class walo ke bachne ke chances 3rd Class se bahut jyada the.
3. **Fare:** Jyada Fare = Jyada Survival Chance.
4. **FamilySize / IsAlone:** Akele travel karne walo ke bachne ke chances kam the.

## Final Conclusion
Feature Engineering aur Pipeline use karne se model ki Accuracy ~80% aayi. Bina Pipeline ke data leakage ka risk rehta hai. Isliye proper Preprocessing + Pipeline hi best practice hai.